## Imports

In [1]:
import pathlib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcodec.encoders import VideoEncoder
from torchvision.transforms import v2

import tqdm
from PIL import Image
from IPython.display import Video

from mobileunet import mobileunet

## Setup

### Parameters

In [2]:
DEVICE = "cuda:0"
CHECKPOINT_PATH = "checkpoints/football_heatmap.pt"

DS_ROOT = "assets/soccernet/SNMOT-116"
BATCH_SIZE = 16
RESIZE_SHAPE = (360, 640)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

FPS = 25

### Helper functions

In [3]:
def unnormalize(x: torch.Tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(x.device)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1).to(x.device)
    return (x * std) + mean


def count_trainable_params(module: nn.Module):
    params = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"Module [{module.__class__.__name__}] trainable parameters: {params:,}")
    return params

In [4]:
def _encode_clip(frames: torch.Tensor):
    if frames.dtype == torch.float32:
        frames = (frames * 255).to(torch.uint8)
    encoder = VideoEncoder(frames, frame_rate=FPS)
    encoded_frames = encoder.to_tensor(format="mp4")
    return encoded_frames


def play_video(frames: torch.Tensor):
    encoded_bytes = _encode_clip(frames)
    return Video(
        data=encoded_bytes.numpy().tobytes(),
        embed=True,
        width=RESIZE_SHAPE[1],
        height=RESIZE_SHAPE[0],
        mimetype="video/mp4",
    )

In [5]:
def mask_frames(frames: torch.Tensor, masks: torch.Tensor, alpha: float = 0.2):
    # frames: [T, C, H, W]
    # masks: [T, D, H, W]
    full_mask = masks.sum(1, keepdim=True)
    full_mask = alpha + (1.0 - alpha) * full_mask
    return frames * full_mask.clamp(0.0, 1.0)

### Load Model

In [6]:
model = mobileunet.mobileunet_large(
    in_channels=3,
    out_channels=7,
    pretrained=True,
).to(DEVICE)
loaded_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
print("Loading checkpoint weights:", model.load_state_dict(loaded_dict["model"]))

_ = model.eval()

Loading checkpoint weights: <All keys matched successfully>


### Load data

In [7]:
class SoccerNetDataset(Dataset[torch.Tensor]):

    def __init__(self, root: str, transforms):
        root_path = pathlib.Path(root)
        self.image_paths = sorted(root_path.glob("*.jpg"))
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index) -> torch.Tensor:
        image_path =self.image_paths[index]
        image = Image.open(image_path)
        return self.transforms(image)

In [8]:
transforms = v2.Compose(
    [
        v2.Resize(size=RESIZE_SHAPE),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float, scale=True),
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
ds = SoccerNetDataset(DS_ROOT, transforms)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

## Run inference

In [9]:
xlist, yhat_list = [], []
for _x in tqdm.tqdm(dl):
    with torch.no_grad():
        _yhat = model(_x.to(DEVICE)).detach().cpu()

    xlist.append(_x)
    yhat_list.append(_yhat)

x = torch.cat(xlist)
yhat = torch.cat(yhat_list)

100%|██████████| 47/47 [00:29<00:00,  1.57it/s]


### Visualize

In [10]:
x_view = unnormalize(x)
masked_frames = mask_frames(x_view, yhat.sigmoid())
play_video(masked_frames)